In [ ]:
#importacion de librerias
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from sklearn.preprocessing import LabelEncoder

In [ ]:
# Cargar fichero
df_final_mejora_1 = pd.read_csv('df_final_mejora_1.csv')

In [ ]:
df_final_mejora_1["fecha_hora"] = pd.to_datetime(df_final_mejora_1["fecha_hora"])

In [ ]:
#vemos los duplicados
df_final_mejora_1[df_final_mejora_1.duplicated(subset=["fecha_hora"], keep=False)]

,provincia,fecha,dia_semana,hora,nivel_triaje,ambito_procedencia,hospital,edad,sexo,dia,...,mes,year,pacientes_ayer,target_pacientes,fecha_hora,hora_redondeada,grupo_edad,grupo_temperatura_media,grupo_precipitaciones,mes_nombre
1,Burgos,2025-01-01,MIÉRCOLES,20:29,3,Urbano,C.A.U. Burgos,82,Mujer,1,...,1,2025,346,346,2025-01-01 20:29:00,20,anciano,mucho-frio,sin_lluvia,Enero
2,Burgos,2025-01-01,MIÉRCOLES,23:16,4,Urbano,H. Santiago Apóstol,4,Hombre,1,...,1,2025,346,346,2025-01-01 23:16:00,23,infante,mucho-frio,sin_lluvia,Enero
3,Burgos,2025-01-01,MIÉRCOLES,1:32,2,Rural,C.A.U. Burgos,60,Mujer,1,...,1,2025,346,346,2025-01-01 01:32:00,2,anciano-joven,mucho-frio,sin_lluvia,Enero
4,Burgos,2025-01-01,MIÉRCOLES,2:38,0,Rural,H. Santos Reyes,26,Mujer,1,...,1,2025,346,346,2025-01-01 02:38:00,3,joven,mucho-frio,sin_lluvia,Enero
5,Burgos,2025-01-01,MIÉRCOLES,18:32,4,Urbano,C.A.U. Burgos,7,Mujer,1,...,1,2025,346,346,2025-01-01 18:32:00,19,infante,mucho-frio,sin_lluvia,Enero
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
782763,Ávila,2024-12-31,MARTES,0:45,4,Urbano,C.A. Ávila,34,Mujer,31,...,12,2024,68,68,2024-12-31 00:45:00,1,adulto,frio,sin_lluvia,Diciembre
782764,Ávila,2024-12-31,MARTES,12:20,5,Urbano,C.A. Ávila,85,Hombre,31,...,12,2024,68,68,2024-12-31 12:20:00,12,anciano,frio,sin_lluvia,Diciembre
782765,Ávila,2024-12-31,MARTES,0:50,4,Urbano,C.A. Ávila,25,Hombre,31,...,12,2024,68,68,2024-12-31 00:50:00,1,joven,frio,sin_lluvia,Diciembre
782768,Ávila,2024-12-31,MARTES,15:20,0,Rural,C.A. Ávila,91,Mujer,31,...,12,2024,68,68,2024-12-31 15:20:00,15,anciano,frio,sin_lluvia,Diciembre


In [ ]:
#eliminamos los duplicados ya que nos queda una buena cantidad de registros para trabajar
df_final_mejora_1 = df_final_mejora_1.drop_duplicates(subset=["fecha_hora"])

In [ ]:
#quitamos las columnas que no son necesarias
df_final_mejora_1 = df_final_mejora_1.drop([
    "fecha","hora", "nivel_triaje", "ambito_procedencia", "edad", "sexo", "grupo_edad", "hora_redondeada", "dia", "grupo_edad", "grupo_temperatura_media", "grupo_precipitaciones", "mes_nombre"
    ], axis=1)

In [ ]:
df_final_mejora_1

,provincia,dia_semana,hospital,temperatura_media,temperatura_maxima,temperatura_minima,viento,viento_maximo,precipitaciones,humedad,mes,year,pacientes_ayer,target_pacientes,fecha_hora
0,Burgos,MIÉRCOLES,H. Santos Reyes,-1.3,4.2,-5.0,8.3,16.5,0.0,97.0,1,2025,191,346,2025-01-01 15:00:00
1,Burgos,MIÉRCOLES,C.A.U. Burgos,-1.3,4.2,-5.0,8.3,16.5,0.0,97.0,1,2025,346,346,2025-01-01 20:29:00
2,Burgos,MIÉRCOLES,H. Santiago Apóstol,-1.3,4.2,-5.0,8.3,16.5,0.0,97.0,1,2025,346,346,2025-01-01 23:16:00
3,Burgos,MIÉRCOLES,C.A.U. Burgos,-1.3,4.2,-5.0,8.3,16.5,0.0,97.0,1,2025,346,346,2025-01-01 01:32:00
4,Burgos,MIÉRCOLES,H. Santos Reyes,-1.3,4.2,-5.0,8.3,16.5,0.0,97.0,1,2025,346,346,2025-01-01 02:38:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
782761,Ávila,MARTES,C.A. Ávila,7.4,13.1,4.0,9.4,18.3,0.0,81.0,12,2024,68,68,2024-12-31 20:01:00
782766,Ávila,MARTES,C.A. Ávila,7.4,13.1,4.0,9.4,18.3,0.0,81.0,12,2024,68,68,2024-12-31 21:30:00
782767,Ávila,MARTES,C.A. Ávila,7.4,13.1,4.0,9.4,18.3,0.0,81.0,12,2024,68,68,2024-12-31 21:15:00
782769,Ávila,MARTES,C.A. Ávila,7.4,13.1,4.0,9.4,18.3,0.0,81.0,12,2024,68,68,2024-12-31 03:35:00


In [ ]:
df = df_final_mejora_1.copy()

In [ ]:
# Extraer solo la fecha
df["fecha"] = df["fecha_hora"].dt.date

# Contar registros por día
pacientes_por_dia = df.groupby("fecha").size()

# Asignar a cada fila el número de pacientes del día anterior
df["pacientes_ayer"] = (
    df["fecha"]
    .map(pacientes_por_dia.shift(1))
)

In [ ]:
# Ordenar por fecha y hora
df = df.sort_values("fecha_hora")

# Extraer solo la fecha
df["fecha"] = df["fecha_hora"].dt.date

# Contador acumulado por día
df["target_pacientes"] = (
    df.groupby("fecha")
      .cumcount() + 1
)

In [ ]:
#en el dia 01/01/2024 no tenemos registro de pacientes ayer asi que lo rellenamos con el dia siguiente para no dejarlo a nulo
df["pacientes_ayer"] = df["pacientes_ayer"].bfill()

In [ ]:
#cambiamos de float a int
df["pacientes_ayer"] = df["pacientes_ayer"].astype(int)

In [ ]:
#quitamos la columna fecha que nos nos hace mas falta
df = df.drop(["fecha"], axis=1)

In [ ]:
#creamos los diferentes mapeos para pasarlo todo a numero y tenerlo todo controlado
mapeo_provincia = {
    "Burgos": 0,
    "León": 1,
    "Salamanca": 2,
    "Segovia": 3,
    "Soria": 4,
    "Valladolid": 5,
    "Zamora": 6,
    "Ávila": 7
}
mapeo_dia_semana = {
    "LUNES": 0,
    "MARTES": 1,
    "MIÉRCOLES": 2,
    "JUEVES": 3,
    "VIERNES": 4,
    "SÁBADO": 5,
    "DOMINGO": 6
}

mapeo_hospital = {
    "C.A.U. Burgos": 0,
    "C.A.U. León": 1,
    "C.A.U. Salamanca": 2,
    "H.U. Río Hortega": 3,
    "H.C.U. Valladolid": 4,
    "H. El Bierzo": 5,
    "H. Santos Reyes": 6,
    "C.A. Zamora": 7,
    "C.A. Segovia": 8,
    "H. Santiago Apóstol": 9,
    "C.A. Ávila": 10,
    "C.A. Soria": 11,
    "H. Medina del Campo": 12
}

df["provincia"] = df["provincia"].map(mapeo_provincia)
df["dia_semana"] = df["dia_semana"].map(mapeo_dia_semana)
df["hospital"] = df["hospital"].map(mapeo_hospital)

In [ ]:
df.to_csv("df_mejora_2.csv", index=False)